# Self-consistency: solving for the equilibrium against assuming one

Two `map_laplace` runs per truth on the same data, seed and priors, differing only in
the geometry the diagnostic points are read on. The fixed arm assumes
`run_study.FIXED_EQUILIBRIUM_ID`, a training-split equilibrium that is deliberately not
the truth, since a real reconstruction cannot know it. The self-consistent arm keeps the
surrogate in the loop, so the pressure the profiles imply predicts the geometry. A
difference between the arms is the geometry alone.

```
python src/.../results/plasma_state_estimation/self_consistency_analysis/run_study.py --workers 4
```

The runner writes `profile_parameters.csv` and `inference_ids.json` beside itself.


In [ ]:
import csv
import pathlib

import numpy as np
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.results.plasma_state_estimation.self_consistency_analysis import (  # noqa: E501
    run_study,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    inference_store_utils,
    pubfig,
    sweep_utils,
)
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    plots_plasma_state_estimation as plots,
)

pubfig.apply_style()

# Anchor every path to the package location, never to the kernel's working directory.
PACKAGE_DIRECTORY = pathlib.Path(plots.__file__).parents[2]
NOTEBOOK_DIRECTORY = (
    PACKAGE_DIRECTORY
    / "results_plots_tables"
    / "results"
    / "plasma_state_estimation"
    / "self_consistency_analysis"
)
PLOTS_DIRECTORY = NOTEBOOK_DIRECTORY / "plots"
STORE = inference_store_utils.INFERENCE_STORE

ARMS = ("fixed_equilibrium", "self_consistent")
INFERENCE_IDS = sweep_utils.current_ids(NOTEBOOK_DIRECTORY / "inference_ids.json")
with (NOTEBOOK_DIRECTORY / "profile_parameters.csv").open() as handle:
    ROWS = list(csv.DictReader(handle))
TRUTHS = list(dict.fromkeys(row["truth_name"] for row in ROWS))
print(f"{len(ROWS)} parameter rows over {len(TRUTHS)} truths: {TRUTHS}")
for label, inference_id in sorted(INFERENCE_IDS.items()):
    print(f"  {label:32s} {inference_id}")

## The table

Each arm's inferred parameters against the target, the truth projected onto the fitted
family. The projection's residual is a floor under both arms. The deviation is signed and relative to the target, so
a target near zero gives a large percentage from a small error: read the target column
beside it.


In [ ]:
print(
    f"{'truth':16s} {'profile':22s} {'k':>2s} {'target':>10s} "
    + "  ".join(f"{arm:>18s}" for arm in ARMS)
)
DEVIATIONS = {arm: [] for arm in ARMS}
TARGETS = []
for truth_name in TRUTHS:
    rows = [row for row in ROWS if row["truth_name"] == truth_name]
    TARGETS.append(np.array([float(row["target"]) for row in rows]))
    for arm in ARMS:
        DEVIATIONS[arm].append(
            np.array(
                [
                    100.0
                    * (float(row[arm]) - float(row["target"]))
                    / abs(float(row["target"]))
                    for row in rows
                ]
            )
        )
    for position, row in enumerate(rows):
        print(
            f"{row['truth_name']:16s} {row['profile']:22s} "
            f"{row['parameter_index']:>2s} {float(row['target']):10.4f} "
            + "  ".join(
                f"{float(row[arm]):10.4f} ({DEVIATIONS[arm][-1][position]:+6.1f}%)"
                for arm in ARMS
            )
        )
DEVIATIONS = {arm: np.stack(values) for arm, values in DEVIATIONS.items()}
print()
for arm in ARMS:
    print(
        f"{arm:22s} median |deviation| {np.median(np.abs(DEVIATIONS[arm])):6.2f}%   "
        f"worst {np.max(np.abs(DEVIATIONS[arm])):7.2f}%"
    )

## What the geometry assumption costs, parameter by parameter

One panel per parameter, the truths along the axis. An arm sitting systematically to one
side carries a bias from its geometry; one only scattered more widely loses precision,
not accuracy.


In [ ]:
profile_labels = (r"$n_e$", r"$T_e$")
per_profile = DEVIATIONS[ARMS[0]].shape[1] // len(profile_labels)
# At most four panels per profile; a longer family shows evenly spaced parameters.
n_columns = min(per_profile, 4)
picks = np.linspace(0, per_profile - 1, n_columns).round().astype(int)
figure, axes_array = pubfig.subplots(
    len(profile_labels), n_columns, width="double", aspect=0.85
)
panels = np.asarray(axes_array).reshape(len(profile_labels), n_columns)
positions = np.arange(len(TRUTHS))
for row, label in enumerate(profile_labels):
    for column, index in enumerate(picks):
        panel = panels[row, column]
        for order, arm in enumerate(ARMS):
            panel.plot(
                positions,
                DEVIATIONS[arm][:, row * per_profile + index],
                "o",
                ms=3.5,
                color=pubfig.PALETTE[order % len(pubfig.PALETTE)],
                label=arm.replace("_", " "),
            )
        panel.axhline(0.0, color="tab:red", ls="--", lw=0.9)
        panel.set_title(f"{label} [{index}]", fontsize=6)
        panel.set_xticks(positions)
        panel.set_xticklabels(
            [name.replace("_", " ") for name in TRUTHS], rotation=90, fontsize=5
        )
        if column == 0:
            panel.set_ylabel("deviation [%]")
panels[0, 0].legend(fontsize=5)
pubfig.save(figure, str(PLOTS_DIRECTORY / "self_consistency_parameter_deviation"))
print("self_consistency_parameter_deviation")

## The same comparison in profile space, one figure per truth

Both arms overlaid, drawn as in `mcmc_benchmark`: each arm's MAP solid and its Laplace 5
to 95% band dashed, against the truth.

In [ ]:
import pandas as pd
from plasma_state_estimation_for_diagnostic_placement_optimization.results_plots_tables.utils import (  # noqa: E501
    profile_table_utils,
)

for truth_name in TRUTHS:
    # Each arm's first noise draw, its only one by default.
    records = {}
    for arm in ARMS:
        labels = sorted(
            label
            for label in INFERENCE_IDS
            if run_study.split_label(label) == (truth_name, arm)
        )
        if labels:
            records[arm] = inference_store_utils.load_record(
                STORE, INFERENCE_IDS[labels[0]]
            )
    if len(records) < len(ARMS):
        print(f"  {truth_name}: incomplete set of arms, skipped")
        continue
    figure = plots.plot_method_band_comparison(
        pd.concat(
            [
                profile_table_utils.profile_band_table(record, arm)
                for arm, record in records.items()
            ]
        ),
        ARMS,
        labels={arm: arm.replace("_", " ") for arm in ARMS},
        title=truth_name,
    )
    pubfig.save(
        figure, str(PLOTS_DIRECTORY / f"self_consistency_profiles_{truth_name}")
    )
    print(f"  self_consistency_profiles_{truth_name}")